# Evaluating models on time series

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/x-datascience-datacamp/datacamp-master/blob/main/06_learning_with_time_series/01b-evaluating_on_time_series.ipynb)

```
Authors: Thomas Moreau
         Pedro Rodrigues
```

Licensed under [CC BY 4.0 Creative Commons](http://creativecommons.org/licenses/by/4.0/)

Cross-validation estimates how a model performs on data it has never seen.
Before choosing a split, ask what this new data will be. For a forecast, it is
the **future** of the series: the model is trained on the past and used on what
comes after. The test folds must reproduce this situation, otherwise the score
answers another question than the one we care about.

With time series, a careless split breaks this in two ways:

1. **Direction**: a random test set puts the future in the training set. Each
   test point sits between training points that are strongly correlated with
   it, and the score measures how well the model interpolates, not how well it
   forecasts. We show it with a single train/test split.
2. **Changing periods**: cross-validation repeats the split along the series.
   When the series is not stationary, the error depends on which period is
   tested, and a single test period gives a noisy estimate.

The notebook ends with other sources of leakage, from features that use data
posterior to the forecast.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path

URL_REPO = "https://github.com/x-datascience-datacamp/datacamp-master/raw/main/06_learning_with_time_series/"


def data_path(name):
    """Local file if the repo is cloned, its GitHub URL otherwise (Colab)."""
    path = Path("data") / name
    return path if path.exists() else f"{URL_REPO}data/{name}"

## 1. Test on the future, not between training points

We use the monthly airline passengers (see `01a`), in log scale so that the
seasonal pattern has a constant amplitude. Our need is to **forecast** the
traffic of the coming months.

Let's start with the usual recipe of supervised learning. The feature `X` is
the time, counted in months, and the target `y` is the value to predict. We
split the data at random into a train and a test set, fit a nearest-neighbors
regressor, which predicts a month from the closest months in the training set,
and score it on the test set.

In [ ]:
df = pd.read_csv(
    data_path("AirPassengers.csv"), parse_dates=["Month"], index_col="Month"
).asfreq("MS")
y = np.log(df["Passengers"])
X = np.arange(len(y)).reshape(-1, 1)

In [ ]:
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor

n_test = 24
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=n_test, random_state=0
)
model = KNeighborsRegressor(n_neighbors=2)
model.fit(X_train, y_train)
y_pred_test = model.predict(X_test)
print(f"RMSE on a random test set: {root_mean_squared_error(y_test, y_pred_test):.3f}")

On a log scale, an RMSE of 0.11 means errors of about 11%. This looks like a
good model. Is anything wrong with this evaluation? Let's look at where the
test months are.

In [ ]:
fig, ax = plt.subplots()
ax.plot(y.index, y, c="lightgray", zorder=0)
ax.scatter(y_train.index, y_train, s=10, label="train")
ax.scatter(y_test.index, y_test, s=10, label="test")
ax.scatter(y_test.index, y_pred_test, marker="x", c="k", label="prediction")
ax.legend();

The test months are scattered over the whole series, and each one sits between
training months. Consecutive months are strongly correlated: on the log series,
the autocorrelation of `01a` is 0.95 at lag 1 and 0.90 at lag 2. A test month is
almost a copy of its training neighbors, so the model only has to
**interpolate**.

A random split assumes i.i.d. samples, and this assumption fails here. More
importantly, the test set does not reproduce our need. A forecast is used on
months that come **after** all the data the model has seen. The test set must
be the new data: we keep the last two years aside, train on the past, and
predict the future.

In [ ]:
X_past, y_past = X[:-n_test], y.iloc[:-n_test]
X_new, y_new = X[-n_test:], y.iloc[-n_test:]

model.fit(X_past, y_past)
y_pred_new = model.predict(X_new)
print(f"RMSE on the new data: {root_mean_squared_error(y_new, y_pred_new):.3f}")

fig, ax = plt.subplots()
ax.plot(y.index, y, label="true")
ax.plot(y_new.index, y_pred_new, label="forecast")
ax.legend();

The error is three times larger. The model cannot forecast: for any month after
1958, its two nearest neighbors are the last two months of the training data,
and the forecast is flat. The random split answered another question than the
one we asked, and it hid this failure.

The right split depends on the use case, not on the data being a time series.
If the task were to fill missing months inside the series (a counter that
broke for a few weeks), the new data would be months surrounded by known ones.
The random split would then reproduce the use case, and a nearest-neighbors
interpolation would be a sensible model.

<div class="alert alert-success">

**EXERCISE:**

- For each test month of both splits, compute the distance, in months, to the closest training month.
- Plot the absolute error against this distance, for both splits. What happens when the forecast goes further from the training data?

</div>

Solution is in: `solutions/eval-1_distance-to-train.py`

In the random split, every test month is one or two months away from a training
month. When forecasting, the distance goes up to two years. The error
oscillates with the seasons, as the flat forecast misses the summer peaks, and
it tends to grow with the distance. The autocorrelation decreases the same way,
from 0.95 at lag 1 to 0.52 at lag 24.

A test set must reproduce the **horizon** of the use case: how far the
predicted values are from the last data the model has seen.

## 2. Cross-validation: repeating the split

A single train/test split tests one period. Cross-validation repeats the split
to test several of them, and the bike counts give us enough data to do it. We
forecast the hourly number of bikes on boulevard de Sébastopol (see `01a` and
`02c`) for the **next 24 hours**, from the **past week** (168 hours).

A common way to build such a dataset, in particular for deep learning, is to
cut the series into **windows**: each sample takes the values before a time $T$
as input, and the values after $T$ as target. We make one window per day.

We predict the log of the counts, so the RMSE below is the RMSLE used in `02c`.

In [ ]:
bikes = pd.read_parquet(data_path("bike-counter-data.parquet"))
counts = (
    bikes.query("counter_name == 'Totem 73 boulevard de Sébastopol S-N'")
    .set_index("date")["bike_count"]
    .sort_index()
)
y = np.log1p(counts.asfreq("h").interpolate()).to_numpy()

In [ ]:
n_past, n_future = 168, 24
origins = np.arange(n_past, len(y) - n_future + 1, 24)  # one window per day

X = np.stack([y[t - n_past:t] for t in origins])
Y = np.stack([y[t:t + n_future] for t in origins])
X.shape, Y.shape

Each window only uses the past of its origin, so no feature leaks the future.
Consecutive windows overlap, though: the 24 target hours of one window are part
of the inputs of the 7 next windows.

The figure below compares splitting strategies for cross-validation. A shuffled
`KFold` tests windows surrounded by training windows, as the random split of
Part 1. `TimeSeriesSplit` only trains on the past of each test fold. Its `gap`
parameter removes samples between the training and the test sets, to mimic a
delay between the last available data and the forecast. Here, windows closer
than 8 days share some hours, so a gap of 8 windows ensures that no training
window overlaps a test window. `max_train_size` limits the training window
(used in `02b`).

In [ ]:
from sklearn.model_selection import KFold, TimeSeriesSplit

splitters = {
    "KFold(shuffle=True)": KFold(n_splits=5, shuffle=True, random_state=0),
    "TimeSeriesSplit()": TimeSeriesSplit(n_splits=5),
    "TimeSeriesSplit(gap=8)": TimeSeriesSplit(n_splits=5, gap=8),
}
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
for ax, (name, splitter) in zip(axes, splitters.items()):
    for k, (train, test) in enumerate(splitter.split(X)):
        ax.scatter(train, [k] * len(train), c="C0", marker="_", lw=8)
        ax.scatter(test, [k] * len(test), c="C1", marker="_", lw=8)
    ax.set_title(name)
    ax.set_ylabel("fold")
axes[-1].set_xlabel("window index (blue: train, orange: test)")
plt.tight_layout()

Let's evaluate a random forest-like model, which predicts the 24 outputs at
once, with each splitter.

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import cross_val_score

model = ExtraTreesRegressor(n_estimators=100, min_samples_leaf=2, random_state=0)

for name, cv in splitters.items():
    scores = -cross_val_score(
        model, X, Y, cv=cv, scoring="neg_root_mean_squared_error"
    )
    print(f"{name}: {scores.mean():.3f} ± {scores.std():.3f}")

As in Part 1, the shuffled split is optimistic: each test window has its
neighbors in the training set, and the model knows the level of the series at
that time (lockdown, holidays, ...). The gap changes little here, because only
the last week of each training set overlaps a test fold of two months.

The shuffled folds also hide something: their scores barely vary (± 0.03),
because each of them mixes windows from all periods. The temporal folds each
test one period, and their scores vary four times more (± 0.12).

<div class="alert alert-success">

**EXERCISE:**

- For `TimeSeriesSplit(gap=8)`, print the score of each fold with the dates of its test period (the origin of window `i` is `dates[origins[i]]`, with `dates = counts.asfreq("h").index`). Which periods are hard to forecast, and why?
- What would a single hold-out on the last two months, as in Part 1, report?

</div>

Solution is in: `solutions/eval-2_sliding-windows.py`

The error on new data depends on **which period comes next**. It goes from 0.56
during the lockdown and curfew of winter 2020 down to 0.23 in September 2021,
when traffic is back to normal.

The series is **not stationary** (see `01a`): its distribution changes over time, with
lockdowns, curfews and holidays. Evaluating on new data assumes that the test
period looks like the future the model will face, and a non-stationary series
breaks this assumption for any single period. A hold-out on the last two months
would report 0.23, less than half the error of the first fold, and nothing
guarantees that the next months will be as easy.

Use several temporal folds and report their mean and spread. The spread is part
of the result: it tells how much the performance changes when the conditions
change.

## Other sources of leakage

The split is not the only way to leak the future. A feature leaks if it uses
data posterior to the forecast:

- a rolling or centered mean that includes the current or future values,
- a filling of missing values with `bfill` or interpolation over the test period,
- external covariates measured after the fact (the observed weather instead of
  the weather forecast available at prediction time),
- a scaler or a target encoder fit on the full series before splitting.

**Checklist**: for each feature, ask *"would I know this value at the moment I
make the forecast?"*, and evaluate with a split that respects time, with a gap
that matches the overlap between samples.